<a href="https://colab.research.google.com/github/bibek0923/AI-based-Nepali-News-Headline-Generator/blob/main/major_projectv01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install -q datasets transformers accelerate evaluate jiwer librosa soundfile
!pip install -q git+https://github.com/openai/whisper.git
!pip install torchcodec


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import pandas as pd
from datasets import Dataset, Audio
import os

# Try reading with comma separator and skipping bad lines
df = pd.read_csv("/content/drive/MyDrive/major_project/devnagari_dataset.csv", sep=',', on_bad_lines='skip', engine='python')
print(df)

# Construct the full path to the audio files using the absolute path to the audio folder
df["audio_filename"] = df["filename"].apply(lambda x: os.path.join("/content/drive/MyDrive/major_project/audio", x))

dataset = Dataset.from_pandas(df)

dataset = dataset.cast_column(
    "audio_filename",
    Audio(sampling_rate=16000)
)

dataset

       filename                                   transcript
0    pk_001.wav       मेरो असाइनमेन्ट सबमिट गर्न टाइम लाग्छ।
1    pk_002.wav              सरले भनेको कन्सेप्ट क्लियर भएन।
2    pk_003.wav          तिमी चाहिँ कति बजे घर जाने प्लान छ?
3    pk_004.wav       क्लासपछि कृपया मलाई पीडीएफ पठाउनुहोस्।
4    pk_005.wav              मेरो फोनको ब्याट्री लगभग सकियो।
..          ...                                          ...
494  st_496.wav                  आजै कोडिङ अभ्यास पूरा गरौँ।
495  st_497.wav                मेरो बाइकको ब्रेकमा समस्या छ।
496  st_498.wav      कृपया यसका लागि नयाँ फोल्डर बनाउनुहोस्।
497  st_499.wav    त्यो डाइग्राम जाँच गर्न सरले भन्नुभएको छ।
498  st_500.wav  मैले चाहिँ अन्ततः त्यो आउटपुट भेरिफाइ गरेँ।

[499 rows x 2 columns]


Dataset({
    features: ['filename', 'transcript', 'audio_filename'],
    num_rows: 499
})

In [7]:
dataset = dataset.rename_column("audio_filename", "audio")
dataset = dataset.rename_column("transcript", "text")


In [8]:
dataset = dataset.train_test_split(test_size=0.1)
train_ds = dataset["train"]
eval_ds = dataset["test"]


In [9]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_NAME = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(
    MODEL_NAME,
    language="ne",
    task="transcribe"
)

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.config.use_cache = False # Explicitly set use_cache to False to prevent RuntimeError during training

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [10]:
def prepare(batch):
    audio = batch["audio"]

    batch["input_features"] = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(
        batch["text"]
    ).input_ids

    return batch


In [11]:
train_ds = train_ds.map(
    prepare,
    remove_columns=train_ds.column_names,
    num_proc=2
)

eval_ds = eval_ds.map(
    prepare,
    remove_columns=eval_ds.column_names,
    num_proc=2
)


Map (num_proc=2):   0%|          | 0/449 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/50 [00:00<?, ? examples/s]

In [12]:
from dataclasses import dataclass
import torch

@dataclass
class WhisperDataCollator:
    processor: WhisperProcessor

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )

        labels = labels_batch["input_ids"]
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        batch["labels"] = labels
        return batch


data_collator = WhisperDataCollator(processor)


In [13]:
import evaluate

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")


In [14]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(
        pred_ids, skip_special_tokens=True
    )

    label_str = processor.tokenizer.batch_decode(
        label_ids, skip_special_tokens=True
    )

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer, "cer": cer}


In [2]:
from transformers import Seq2SeqTrainingArguments

# training_args = Seq2SeqTrainingArguments(
#     output_dir="./whisper-nepali",
#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=8,
#     gradient_accumulation_steps=2,
#     learning_rate=1e-5,
#     warmup_steps=500,
#     max_steps=2000,
#     gradient_checkpointing=False, # Changed to False to fix the RuntimeError
#     fp16=True,
#     eval_strategy="steps",
#     eval_steps=100,
#     save_steps=100,
#     logging_steps=100,
#     save_total_limit=2,
#     report_to="none",
#     load_best_model_at_end=True,
#     metric_for_best_model="wer",
#     greater_is_better=False
# )
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=5e-5,
    # learning_rate=1e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    save_strategy="epoch",
    save_total_limit=2,
    predict_with_generate=True,
    fp16=True,
    logging_dir="./logs",
    logging_steps=200,
    load_best_model_at_end=True
)

In [15]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor
)


/tmp/ipython-input-3656997460.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [16]:
trainer.train()


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Wer,Cer
1,No log,0.922835,13.481928,10.614994
2,1.089600,0.671374,1.052209,7.039390
3,1.089600,0.648504,0.955823,7.104193
4,0.316800,0.641166,0.775100,6.961245
5,0.316800,0.628653,6.975904,7.881194
6,0.136100,0.629931,1.068273,6.903431
7,0.136100,0.648586,3.024096,7.519695
8,0.050800,0.656349,6.365462,8.001271
9,0.014000,0.641196,4.100402,7.602287
10,0.014000,0.642194,2.967871,7.339263


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}. You are seeing thi

TrainOutput(global_step=1130, training_loss=0.2847235477339905, metrics={'train_runtime': 3293.1016, 'train_samples_per_second': 1.363, 'train_steps_per_second': 0.343, 'total_flos': 1.2957484474368e+18, 'train_loss': 0.2847235477339905, 'epoch': 10.0})

In [17]:
trainer.save_model("./whisper-nepali-final")
processor.save_pretrained("./whisper-nepali-final")


[]

In [18]:
!zip -r whisper-nepali-final.zip whisper-nepali-final


  adding: whisper-nepali-final/ (stored 0%)
  adding: whisper-nepali-final/special_tokens_map.json (deflated 80%)
  adding: whisper-nepali-final/added_tokens.json (deflated 80%)
  adding: whisper-nepali-final/tokenizer_config.json (deflated 96%)
  adding: whisper-nepali-final/training_args.bin (deflated 54%)
  adding: whisper-nepali-final/config.json (deflated 60%)
  adding: whisper-nepali-final/merges.txt (deflated 54%)
  adding: whisper-nepali-final/normalizer.json (deflated 81%)
  adding: whisper-nepali-final/vocab.json (deflated 58%)
  adding: whisper-nepali-final/model.safetensors (deflated 8%)
  adding: whisper-nepali-final/tokenizer.json (deflated 82%)
  adding: whisper-nepali-final/preprocessor_config.json (deflated 44%)
  adding: whisper-nepali-final/generation_config.json (deflated 73%)


In [19]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

processor = WhisperProcessor.from_pretrained("./whisper-nepali-final")
model = WhisperForConditionalGeneration.from_pretrained("./whisper-nepali-final")

model.eval()


WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 768)
      (layers): ModuleList(
        (0-11): 12 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (f